# Use an AMD RX 5700 XT from PyTorch

[English](install_pytorch_mps_rx5700xt.ipynb) | [简体中文](install_pytorch_mps_rx5700xt_CN.ipynb)

Yes, this works. On the Ryzen 9 3900X Hackintosh used for this tutorial,
PyTorch accesses the RX 5700 XT through Apple's **MPS/Metal** backend and shows
it as `mps:0`.

You only need to do three things:

1. install the tested Python and PyTorch versions;
2. confirm that `mps:0` is available;
3. run a short matrix test to make sure the GPU is doing real work.

CUDA and ROCm are not used on macOS.

## Before installing anything

This notebook is for an **x86_64 Hackintosh**, not Apple Silicon. It was tested
with the following setup:

| Item | Configuration |
|---|---|
| CPU / GPU | Ryzen 9 3900X / Radeon RX 5700 XT 8 GB |
| macOS | Sequoia 15.2, x86_64, Metal 3 |
| Python | 3.11 |
| PyTorch | 2.2.2 |
| torchvision / torchaudio | 0.17.2 / 2.2.2 |
| NumPy | 1.26.4 |

PyTorch 2.2.x was the final release line with official macOS x86_64 packages,
which is why this tutorial uses these older versions. Install them in a separate
environment instead of changing the system Python.

Clone the repository and enter this directory:

```bash
git clone https://github.com/hermanzhaozzzz/My-Opencore-EFI-for-AMD3900X-5700XT-TUF-x570-Hackintosh.git
cd My-Opencore-EFI-for-AMD3900X-5700XT-TUF-x570-Hackintosh/tutorial_pytorch-via-this_AMD5700xtGPU
```

Choose **either `uv` or Conda** below. Do not combine both methods in the same
environment.

References: [PyTorch 2.2 release notes](https://pytorch.org/blog/pytorch2-2/) ·
[PyTorch MPS documentation](https://docs.pytorch.org/docs/stable/notes/mps.html)

## Install with `uv` (recommended)

If `uv` is not installed yet:

```bash
brew install uv
```

Create the environment and install the tested packages:

```bash
uv python install 3.11
uv venv --python 3.11 .venv

uv pip install --python .venv/bin/python \
  "numpy==1.26.4" \
  "torch==2.2.2" \
  "torchvision==0.17.2" \
  "torchaudio==2.2.2" \
  "jupyterlab>=4,<5" \
  "ipykernel>=6,<8"
```

Start JupyterLab with the environment you just created:

```bash
.venv/bin/jupyter lab
```

Open `install_pytorch_mps_rx5700xt.ipynb`. Calling `.venv/bin/jupyter` directly
prevents Jupyter from silently using another Python installation.

## Or install with Conda

Use this method instead if you already work with Anaconda or Miniconda:

```bash
conda create -n pytorch-mps-rx5700xt python=3.11 pip -y
conda activate pytorch-mps-rx5700xt

python -m pip install \
  "numpy==1.26.4" \
  "torch==2.2.2" \
  "torchvision==0.17.2" \
  "torchaudio==2.2.2" \
  "jupyterlab>=4,<5" \
  "ipykernel>=6,<8"
```

Then start JupyterLab normally:

```bash
jupyter lab
```

Do not upgrade this environment to PyTorch 2.3 or later: those releases do not
provide the macOS x86_64 package needed by this machine.

## 1. Check the environment

In JupyterLab, choose **Kernel → Restart Kernel and Run All Cells**. A correct
setup reports Python 3.11, x86_64, PyTorch 2.2.2, and `True` for both `MPS built`
and `MPS available`.

For this strict GPU check, CPU fallback must be disabled; normal use does not
require `env -u PYTORCH_ENABLE_MPS_FALLBACK`.

In [1]:
import os
import platform
import sys

import numpy as np
import torch

print(f"Python executable : {os.path.basename(sys.executable)}")
print(f"Python version    : {platform.python_version()}")
print(f"Architecture      : {platform.machine()}")
print(f"NumPy             : {np.__version__}")
print(f"PyTorch           : {torch.__version__}")
print(f"MPS built         : {torch.backends.mps.is_built()}")
print(f"MPS available     : {torch.backends.mps.is_available()}")

assert platform.machine() == "x86_64", "This tutorial was validated on x86_64 only"
assert platform.python_version_tuple()[:2] == ("3", "11"), "Use Python 3.11"
assert torch.__version__.startswith("2.2.2"), "Install PyTorch 2.2.2"
assert torch.backends.mps.is_built(), "This PyTorch build has no MPS support"
assert torch.backends.mps.is_available(), "No available MPS device was found"

fallback = os.environ.get("PYTORCH_ENABLE_MPS_FALLBACK", "").lower()
assert fallback not in {"1", "true", "yes"}, (
    "Disable PYTORCH_ENABLE_MPS_FALLBACK and restart JupyterLab"
)

print("\nEnvironment check passed. You can continue.")

Python executable : python
Python version    : 3.11.15
Architecture      : x86_64
NumPy             : 1.26.4
PyTorch           : 2.2.2
MPS built         : True
MPS available     : True

Environment check passed. You can continue.


## 2. Confirm that PyTorch is using the RX 5700 XT

The next cell creates a tensor on MPS and performs a small calculation. The
important part of the result is:

```text
PASS ... device=mps:0
```

This computer has no integrated GPU, so `mps:0` is the RX 5700 XT. Seeing this
result confirms that the calculation ran through Metal on the graphics card,
not on the Ryzen CPU.

In [2]:
# Copy this minimal verification into another project if needed
import torch

assert torch.backends.mps.is_available(), "MPS is unavailable"

device = torch.device("mps")
x = torch.tensor([1.0, 2.0, 3.0], device=device)
y = x.square()
torch.mps.synchronize()

assert y.tolist() == [1.0, 4.0, 9.0]
print(f"PASS: {y} · device={y.device}")

PASS: tensor([1., 4., 9.], device='mps:0') · device=mps:0


## 3. Run a short performance test

The next cell performs ten FP16 matrix multiplications at `4096 × 4096`. It
usually finishes in a few seconds, after two untimed warm-up passes.

The result is useful as a quick sanity check, not as a universal GPU score.
Background tasks, GPU clocks and cooling will change the number. You can keep
Activity Monitor, iStat Menus or another monitor open to watch GPU load and
temperature while it runs.

In [3]:
import time

matrix_size = 4096
matrix_a = torch.randn(
    (matrix_size, matrix_size), device="mps", dtype=torch.float16
)
matrix_b = torch.randn(
    (matrix_size, matrix_size), device="mps", dtype=torch.float16
)

# Warm up twice so the initial Metal kernel compilation is not timed
torch.mps.synchronize()
for _ in range(2):
    result = matrix_a @ matrix_b
torch.mps.synchronize()

timed_iterations = 10
started = time.perf_counter()
for _ in range(timed_iterations):
    result = matrix_a @ matrix_b
torch.mps.synchronize()
elapsed = time.perf_counter() - started

operations = 2 * matrix_size ** 3 * timed_iterations
tflops = operations / elapsed / 1e12

print(f"Matrix       : {matrix_size} x {matrix_size}")
print(f"Device       : {result.device}")
print(f"Iterations   : {timed_iterations}")
print(f"Elapsed      : {elapsed:.4f} seconds")
print(f"Throughput   : {tflops:.2f} TFLOP/s")
print(f"Checksum     : {result[0, 0].item():.6g}")

assert str(result.device) == "mps:0"
assert tflops > 0

del matrix_a, matrix_b, result
torch.mps.empty_cache()

Matrix       : 4096 x 4096
Device       : mps:0
Iterations   : 10
Elapsed      : 0.2159 seconds
Throughput   : 6.36 TFLOP/s
Checksum     : -19.2656


## Results from this machine

These are the results from the Ryzen 9 3900X and RX 5700 XT used for this EFI:

| Test | Measured result |
|---|---:|
| Minimal MPS check | Passed on `mps:0` |
| Notebook short test | **6.36 TFLOP/s** |
| Five-minute stress-test duration | 300.74 seconds |
| Matrix multiplications completed | 526 |
| Five-minute average | **6.49 TFLOP/s** |
| GPU utilization | About 95%–100% |
| Maximum GPU die temperature | 97°C |
| MPS / driver memory | About 0.84 / 1.13 GiB |

`mps:0` is the real pass condition. The TFLOP/s number is only a reference for
this particular matrix operation and should not be treated as expected model
training speed.

The sustained load confirms that the RX 5700 XT can be driven fully through
MPS. The observed 97°C die temperature is high; improve cooling before running
training jobs for hours at a time.

## Use MPS in your own project

Move both the model and its inputs to the same device:

```python
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = model.to(device)
inputs = inputs.to(device)
outputs = model(inputs)
```

If PyTorch says the model and input are on different devices, check:

```python
print(next(model.parameters()).device)
print(inputs.device)
```

A few practical points:

- The RX 5700 XT has 8 GB of VRAM. Reduce `batch_size` or input dimensions if
  memory runs out.
- Start with `float32`. Try `float16` only after the model works correctly.
- PyTorch 2.2.2 on AMD MPS does not support `bfloat16`.
- Some operations have no MPS implementation. Setting
  `PYTORCH_ENABLE_MPS_FALLBACK=1` lets those operations run on the CPU, but may
  make the workload much slower.
- When checking GPU acceleration, keep fallback disabled and confirm that your
  tensors report `mps:0`.

At this point the environment is ready for normal PyTorch projects.